###1) Konfiguracja

In [0]:
from pathlib import Path
from datetime import (
    datetime,
    date,
    time,
    timezone,
    timedelta,
)
from zoneinfo import ZoneInfo
from zipfile import (
    ZipFile,
    BadZipFile,
)
from io import TextIOWrapper
from collections import Counter
import csv
import uuid

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
)

spark.conf.set(
    "spark.sql.session.timeZone",
    "UTC"
)

dbutils.widgets.text(
    "evaluation_start_date_local",
    "2024-07-01"
)

dbutils.widgets.text(
    "evaluation_end_date_local_exclusive",
    "2024-11-01"
)

dbutils.widgets.text(
    "fmm_history_buffer_days",
    "1"
)

EVALUATION_START_DATE_TEXT = dbutils.widgets.get(
    "evaluation_start_date_local"
)

EVALUATION_END_DATE_TEXT = dbutils.widgets.get(
    "evaluation_end_date_local_exclusive"
)

FMM_HISTORY_BUFFER_DAYS = int(
    dbutils.widgets.get(
        "fmm_history_buffer_days"
    )
)

MARKET_TIMEZONE = "America/Los_Angeles"
LOCAL_TIMEZONE = ZoneInfo(MARKET_TIMEZONE)

evaluation_start_local = datetime.combine(
    date.fromisoformat(
        EVALUATION_START_DATE_TEXT
    ),
    time.min,
    tzinfo=LOCAL_TIMEZONE
)

evaluation_end_local = datetime.combine(
    date.fromisoformat(
        EVALUATION_END_DATE_TEXT
    ),
    time.min,
    tzinfo=LOCAL_TIMEZONE
)

if evaluation_end_local <= evaluation_start_local:
    raise ValueError(
        "Data końcowa musi być późniejsza "
        "od daty początkowej."
    )

EVALUATION_START_UTC = (
    evaluation_start_local
    .astimezone(timezone.utc)
)

EVALUATION_END_UTC = (
    evaluation_end_local
    .astimezone(timezone.utc)
)

# START_UTC i END_UTC dotyczą dostaw DAM
# oraz docelowego okresu analizy.
START_UTC = EVALUATION_START_UTC
END_UTC = EVALUATION_END_UTC

# FMM zawiera dodatkowy dzień historii.
FMM_START_UTC = (
    EVALUATION_START_UTC
    - timedelta(
        days=FMM_HISTORY_BUFFER_DAYS
    )
)

FMM_END_UTC = EVALUATION_END_UTC

CAISO_NODE = "TH_NP15_GEN-APND"

BASE_DIRECTORY = Path(
    "/Volumes/workspace/energy_bronze/"
    "landing/caiso/np15"
)

DAM_DIRECTORY = BASE_DIRECTORY / "dam"
FMM_DIRECTORY = BASE_DIRECTORY / "fmm"

TARGET_TABLE = (
    "workspace.energy_bronze."
    "caiso_lmp_raw"
)

EXPECTED_DAM_ARCHIVES = int(
    (
        END_UTC - START_UTC
    ).total_seconds()
    / (24 * 3600)
)

EXPECTED_FMM_ARCHIVES = int(
    (
        FMM_END_UTC - FMM_START_UTC
    ).total_seconds()
    / 3600
)

EXPECTED_DAM_INTERVALS = int(
    (
        END_UTC - START_UTC
    ).total_seconds()
    / 3600
)

EXPECTED_FMM_INTERVALS = int(
    (
        FMM_END_UTC - FMM_START_UTC
    ).total_seconds()
    / (15 * 60)
)

print("Konfiguracja Bronze CAISO gotowa.")
print(f"Węzeł: {CAISO_NODE}")

print(
    "DAM:",
    START_UTC,
    "—",
    END_UTC
)

print(
    "FMM:",
    FMM_START_UTC,
    "—",
    FMM_END_UTC
)

print(
    "Oczekiwane archiwa DAM:",
    EXPECTED_DAM_ARCHIVES
)

print(
    "Oczekiwane archiwa FMM:",
    EXPECTED_FMM_ARCHIVES
)

print(
    "Oczekiwane interwały DAM:",
    EXPECTED_DAM_INTERVALS
)

print(
    "Oczekiwane interwały FMM:",
    EXPECTED_FMM_INTERVALS
)

print(f"Tabela docelowa: {TARGET_TABLE}")

###2) Znalezienie i kontrola archiwów

In [0]:
archive_specs = []


# DAM: jeden plik na dobę
current_start = START_UTC

while current_start < END_UTC:
    current_end = min(
        current_start + timedelta(days=1),
        END_UTC
    )

    archive_path = DAM_DIRECTORY / (
        "dam_"
        f"{current_start.strftime('%Y%m%dT%H%MZ')}_"
        f"{current_end.strftime('%Y%m%dT%H%MZ')}"
        ".zip"
    )

    archive_specs.append({
        "market_type": "DAM",
        "archive_path": archive_path,
        "request_start_utc": current_start,
        "request_end_utc": current_end,
    })

    current_start = current_end


# FMM: jeden plik na godzinę
current_start = FMM_START_UTC

while current_start < FMM_END_UTC:
    current_end = min(
        current_start + timedelta(hours=1),
        FMM_END_UTC
    )

    archive_path = FMM_DIRECTORY / (
        "fmm_"
        f"{current_start.strftime('%Y%m%dT%H%MZ')}"
        ".zip"
    )

    archive_specs.append({
        "market_type": "FMM",
        "archive_path": archive_path,
        "request_start_utc": current_start,
        "request_end_utc": current_end,
    })

    current_start = current_end


def inspect_archive(spec):
    archive_path = spec["archive_path"]

    if not archive_path.exists():
        return {
            **spec,
            "valid": False,
            "csv_count": 0,
            "error": "Plik nie istnieje",
        }

    if archive_path.stat().st_size == 0:
        return {
            **spec,
            "valid": False,
            "csv_count": 0,
            "error": "Plik jest pusty",
        }

    try:
        with ZipFile(
            archive_path,
            "r"
        ) as archive:
            bad_member = archive.testzip()

            if bad_member is not None:
                return {
                    **spec,
                    "valid": False,
                    "csv_count": 0,
                    "error": (
                        "Uszkodzony element ZIP: "
                        f"{bad_member}"
                    ),
                }

            csv_names = [
                name
                for name in archive.namelist()
                if name.lower().endswith(".csv")
            ]

            if not csv_names:
                return {
                    **spec,
                    "valid": False,
                    "csv_count": 0,
                    "error": "ZIP nie zawiera CSV",
                }

            return {
                **spec,
                "valid": True,
                "csv_count": len(csv_names),
                "error": "",
            }

    except (BadZipFile, OSError) as error:
        return {
            **spec,
            "valid": False,
            "csv_count": 0,
            "error": (
                f"{type(error).__name__}: "
                f"{str(error)}"
            ),
        }


archive_inspection = [
    inspect_archive(spec)
    for spec in archive_specs
]

invalid_archives = [
    item
    for item in archive_inspection
    if not item["valid"]
]

dam_archive_count = sum(
    1
    for item in archive_inspection
    if (
        item["valid"]
        and item["market_type"] == "DAM"
    )
)

fmm_archive_count = sum(
    1
    for item in archive_inspection
    if (
        item["valid"]
        and item["market_type"] == "FMM"
    )
)


print(
    f"Poprawne archiwa DAM: "
    f"{dam_archive_count}/"
    f"{EXPECTED_DAM_ARCHIVES}"
)

print(
    f"Poprawne archiwa FMM: "
    f"{fmm_archive_count}/"
    f"{EXPECTED_FMM_ARCHIVES}"
)

print(
    f"Niepoprawne archiwa: "
    f"{len(invalid_archives)}"
)


if invalid_archives:
    for item in invalid_archives[:10]:
        print(
            f"{item['archive_path'].name}: "
            f"{item['error']}"
        )

    raise RuntimeError(
        "Nie wszystkie archiwa są poprawne."
    )


if (
    dam_archive_count
        != EXPECTED_DAM_ARCHIVES
    or fmm_archive_count
        != EXPECTED_FMM_ARCHIVES
):
    raise RuntimeError(
        "Liczba archiwów nie zgadza się "
        "z oczekiwanym zakresem."
    )


print(
    "Kontrola wejścia zakończona."
)

###3 Odczyt CSV ze wszystkim ZIP-ów

In [0]:
def clean_csv_value(value):
    if value is None:
        return None

    cleaned = str(value).strip()

    if cleaned == "":
        return None

    return cleaned


raw_rows = []


for spec in archive_specs:
    archive_path = spec["archive_path"]
    market_type = spec["market_type"]

    with ZipFile(
        archive_path,
        "r"
    ) as archive:
        csv_names = [
            name
            for name in archive.namelist()
            if name.lower().endswith(".csv")
        ]

        for csv_name in csv_names:
            with archive.open(
                csv_name,
                "r"
            ) as binary_file:
                with TextIOWrapper(
                    binary_file,
                    encoding="utf-8-sig",
                    newline=""
                ) as text_file:
                    reader = csv.DictReader(
                        text_file
                    )

                    if not reader.fieldnames:
                        raise RuntimeError(
                            "Brak nagłówka CSV: "
                            f"{archive_path.name} / "
                            f"{csv_name}"
                        )

                    for source_row in reader:
                        normalized_row = {
                            str(key).strip().upper():
                                clean_csv_value(value)
                            for key, value
                            in source_row.items()
                            if key is not None
                        }

                        interval_start = (
                            normalized_row.get(
                                "INTERVALSTARTTIME_GMT"
                            )
                        )

                        if interval_start is None:
                            continue

                        if market_type == "DAM":
                            price_value = (
                                normalized_row.get("MW")
                            )
                            raw_value_column = "MW"
                            dataset_name = (
                                "caiso_dam_lmp"
                            )
                        else:
                            price_value = (
                                normalized_row.get("PRC")
                            )
                            raw_value_column = "PRC"
                            dataset_name = (
                                "caiso_fmm_lmp"
                            )

                        # Zabezpieczenie na wypadek
                        # innego wariantu nagłówka.
                        if price_value is None:
                            price_value = (
                                normalized_row.get("PRC")
                                or normalized_row.get(
                                    "MW"
                                )
                            )

                        raw_rows.append({
                            "market_type":
                                market_type,
                            "interval_starttime_gmt_raw":
                                interval_start,
                            "interval_endtime_gmt_raw":
                                normalized_row.get(
                                    "INTERVALENDTIME_GMT"
                                ),
                            "opr_dt_raw":
                                normalized_row.get(
                                    "OPR_DT"
                                ),
                            "opr_hr_raw":
                                normalized_row.get(
                                    "OPR_HR"
                                ),
                            "opr_interval_raw":
                                normalized_row.get(
                                    "OPR_INTERVAL"
                                ),
                            "node_id_xml":
                                normalized_row.get(
                                    "NODE_ID_XML"
                                ),
                            "node_id":
                                normalized_row.get(
                                    "NODE_ID"
                                ),
                            "node":
                                normalized_row.get(
                                    "NODE"
                                ),
                            "market_run_id":
                                normalized_row.get(
                                    "MARKET_RUN_ID"
                                ),
                            "lmp_type":
                                normalized_row.get(
                                    "LMP_TYPE"
                                ),
                            "xml_data_item":
                                normalized_row.get(
                                    "XML_DATA_ITEM"
                                ),
                            "pnode_resmrid":
                                normalized_row.get(
                                    "PNODE_RESMRID"
                                ),
                            "grp_type":
                                normalized_row.get(
                                    "GRP_TYPE"
                                ),
                            "pos_raw":
                                normalized_row.get(
                                    "POS"
                                ),
                            "mw_raw":
                                normalized_row.get(
                                    "MW"
                                ),
                            "prc_raw":
                                normalized_row.get(
                                    "PRC"
                                ),
                            "group_raw":
                                normalized_row.get(
                                    "GROUP"
                                ),
                            "price_value_raw":
                                price_value,
                            "raw_value_column":
                                raw_value_column,
                            "request_start_utc_raw":
                                spec[
                                    "request_start_utc"
                                ].isoformat(),
                            "request_end_utc_raw":
                                spec[
                                    "request_end_utc"
                                ].isoformat(),
                            "_source_file_path":
                                str(archive_path),
                            "_source_file_name":
                                archive_path.name,
                            "_source_file_size":
                                int(
                                    archive_path.stat()
                                    .st_size
                                ),
                            "_source_csv_name":
                                csv_name,
                            "_dataset_name":
                                dataset_name,
                            "_source_system":
                                "CAISO_OASIS",
                            "_requested_node":
                                CAISO_NODE,
                        })


row_counts_by_market = Counter(
    row["market_type"]
    for row in raw_rows
)


print(
    f"Utworzono rekordów Python: "
    f"{len(raw_rows)}"
)

print(
    f"Rekordy DAM: "
    f"{row_counts_by_market['DAM']}"
)

print(
    f"Rekordy FMM: "
    f"{row_counts_by_market['FMM']}"
)


if not raw_rows:
    raise RuntimeError(
        "Nie utworzono żadnych rekordów."
    )

###4) Utworzenie DataFrame Bronze

In [0]:
bronze_schema = StructType([
    StructField(
        "market_type",
        StringType(),
        False
    ),
    StructField(
        "interval_starttime_gmt_raw",
        StringType(),
        True
    ),
    StructField(
        "interval_endtime_gmt_raw",
        StringType(),
        True
    ),
    StructField(
        "opr_dt_raw",
        StringType(),
        True
    ),
    StructField(
        "opr_hr_raw",
        StringType(),
        True
    ),
    StructField(
        "opr_interval_raw",
        StringType(),
        True
    ),
    StructField(
        "node_id_xml",
        StringType(),
        True
    ),
    StructField(
        "node_id",
        StringType(),
        True
    ),
    StructField(
        "node",
        StringType(),
        True
    ),
    StructField(
        "market_run_id",
        StringType(),
        True
    ),
    StructField(
        "lmp_type",
        StringType(),
        True
    ),
    StructField(
        "xml_data_item",
        StringType(),
        True
    ),
    StructField(
        "pnode_resmrid",
        StringType(),
        True
    ),
    StructField(
        "grp_type",
        StringType(),
        True
    ),
    StructField(
        "pos_raw",
        StringType(),
        True
    ),
    StructField(
        "mw_raw",
        StringType(),
        True
    ),
    StructField(
        "prc_raw",
        StringType(),
        True
    ),
    StructField(
        "group_raw",
        StringType(),
        True
    ),
    StructField(
        "price_value_raw",
        StringType(),
        True
    ),
    StructField(
        "raw_value_column",
        StringType(),
        True
    ),
    StructField(
        "request_start_utc_raw",
        StringType(),
        True
    ),
    StructField(
        "request_end_utc_raw",
        StringType(),
        True
    ),
    StructField(
        "_source_file_path",
        StringType(),
        False
    ),
    StructField(
        "_source_file_name",
        StringType(),
        False
    ),
    StructField(
        "_source_file_size",
        LongType(),
        False
    ),
    StructField(
        "_source_csv_name",
        StringType(),
        False
    ),
    StructField(
        "_dataset_name",
        StringType(),
        False
    ),
    StructField(
        "_source_system",
        StringType(),
        False
    ),
    StructField(
        "_requested_node",
        StringType(),
        False
    ),
])


ingestion_run_id = str(uuid.uuid4())


raw_caiso_df = spark.createDataFrame(
    raw_rows,
    schema=bronze_schema
)


hash_columns = [
    "market_type",
    "interval_starttime_gmt_raw",
    "interval_endtime_gmt_raw",
    "opr_dt_raw",
    "opr_hr_raw",
    "opr_interval_raw",
    "node_id_xml",
    "node_id",
    "node",
    "market_run_id",
    "lmp_type",
    "xml_data_item",
    "pnode_resmrid",
    "grp_type",
    "pos_raw",
    "mw_raw",
    "prc_raw",
    "group_raw",
]


hash_expression = F.concat_ws(
    "||",
    *[
        F.coalesce(
            F.col(column_name),
            F.lit("<NULL>")
        )
        for column_name in hash_columns
    ]
)


caiso_bronze_df = (
    raw_caiso_df
    .withColumn(
        "_ingestion_run_id",
        F.lit(ingestion_run_id)
    )
    .withColumn(
        "_ingested_at_utc",
        F.current_timestamp()
    )
    .withColumn(
        "_record_hash",
        F.sha2(
            hash_expression,
            256
        )
    )
    .dropDuplicates([
        "_record_hash"
    ])
)


raw_count = raw_caiso_df.count()
bronze_count = caiso_bronze_df.count()


print(f"Rekordy przed deduplikacją: {raw_count}")
print(f"Rekordy po deduplikacji: {bronze_count}")
print(
    "Usunięte dokładne duplikaty: "
    f"{raw_count - bronze_count}"
)
print(f"Ingestion run ID: {ingestion_run_id}")


display(
    caiso_bronze_df
    .orderBy(
        "market_type",
        "interval_starttime_gmt_raw",
        "xml_data_item"
    )
    .limit(20)
)

###5) Zapis tabeli Delta Bronze

In [0]:
spark.sql(
    "CREATE SCHEMA IF NOT EXISTS "
    "workspace.energy_bronze"
)


(
    caiso_bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        TARGET_TABLE
    )
)


saved_row_count = (
    spark.table(TARGET_TABLE).count()
)


print(
    f"Zapisano tabelę: {TARGET_TABLE}"
)

print(
    f"Liczba zapisanych rekordów: "
    f"{saved_row_count}"
)

###6) Profil tabeli

In [0]:
caiso_profile_df = spark.sql(
    f"""
    SELECT
        market_type,
        market_run_id,
        COUNT(*) AS row_count,
        COUNT(
            DISTINCT interval_starttime_gmt_raw
        ) AS distinct_interval_count,
        COUNT(
            DISTINCT _source_file_name
        ) AS source_archive_count,
        MIN(
            interval_starttime_gmt_raw
        ) AS min_interval_start_raw,
        MAX(
            interval_starttime_gmt_raw
        ) AS max_interval_start_raw,
        SUM(
            CASE
                WHEN price_value_raw IS NULL
                THEN 1
                ELSE 0
            END
        ) AS missing_price_count
    FROM {TARGET_TABLE}
    GROUP BY
        market_type,
        market_run_id
    ORDER BY
        market_type
    """
)


display(caiso_profile_df)

###7) Składniki ceny CAISO

In [0]:
caiso_components_df = spark.sql(
    f"""
    SELECT
        market_type,
        market_run_id,
        lmp_type,
        xml_data_item,
        raw_value_column,
        COUNT(*) AS row_count,
        MIN(
            TRY_CAST(
                price_value_raw AS DOUBLE
            )
        ) AS minimum_value,
        MAX(
            TRY_CAST(
                price_value_raw AS DOUBLE
            )
        ) AS maximum_value
    FROM {TARGET_TABLE}
    GROUP BY
        market_type,
        market_run_id,
        lmp_type,
        xml_data_item,
        raw_value_column
    ORDER BY
        market_type,
        xml_data_item
    """
)


display(caiso_components_df)

###8) Testy jakości Bronze

In [0]:
caiso_table_df = (
    spark.table(TARGET_TABLE)
    .withColumn(
        "interval_start_utc",
        F.to_timestamp(
            "interval_starttime_gmt_raw",
            "yyyy-MM-dd'T'HH:mm:ssXXX"
        )
    )
    .withColumn(
        "interval_end_utc",
        F.to_timestamp(
            "interval_endtime_gmt_raw",
            "yyyy-MM-dd'T'HH:mm:ssXXX"
        )
    )
    .withColumn(
        "price_value",
        F.expr(
            "TRY_CAST("
            "price_value_raw AS DOUBLE)"
        )
    )
)


dam_df = caiso_table_df.filter(
    F.col("market_type") == "DAM"
)

fmm_df = caiso_table_df.filter(
    F.col("market_type") == "FMM"
)


dam_archive_count = (
    dam_df
    .select("_source_file_name")
    .distinct()
    .count()
)

fmm_archive_count = (
    fmm_df
    .select("_source_file_name")
    .distinct()
    .count()
)


dam_interval_count = (
    dam_df
    .select("interval_start_utc")
    .distinct()
    .count()
)

fmm_interval_count = (
    fmm_df
    .select("interval_start_utc")
    .distinct()
    .count()
)


invalid_timestamp_count = (
    caiso_table_df
    .filter(
        F.col("interval_start_utc").isNull()
        | F.col("interval_end_utc").isNull()
    )
    .count()
)


invalid_price_count = (
    caiso_table_df
    .filter(
        F.col("price_value_raw").isNotNull()
        & F.col("price_value").isNull()
    )
    .count()
)


duplicate_hash_count = (
    caiso_table_df
    .groupBy("_record_hash")
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)


business_key_columns = [
    "market_type",
    "interval_starttime_gmt_raw",
    "interval_endtime_gmt_raw",
    "node_id",
    "market_run_id",
    "lmp_type",
    "xml_data_item",
    "pnode_resmrid",
    "grp_type",
    "pos_raw",
    "opr_interval_raw",
    "group_raw",
]


duplicate_business_key_count = (
    caiso_table_df
    .groupBy(
        *business_key_columns
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)


invalid_duration_count = (
    caiso_table_df
    .withColumn(
        "duration_minutes",
        (
            F.col("interval_end_utc")
            .cast("long")
            - F.col("interval_start_utc")
            .cast("long")
        ) / F.lit(60.0)
    )
    .filter(
        (
            (F.col("market_type") == "DAM")
            & (
                F.col("duration_minutes")
                != F.lit(60.0)
            )
        )
        |
        (
            (F.col("market_type") == "FMM")
            & (
                F.col("duration_minutes")
                != F.lit(15.0)
            )
        )
    )
    .count()
)


main_lmp_df = caiso_table_df.filter(
    F.upper(
        F.col("xml_data_item")
    ) == "LMP_PRC"
)


dam_main_lmp_interval_count = (
    main_lmp_df
    .filter(
        F.col("market_type") == "DAM"
    )
    .select("interval_start_utc")
    .distinct()
    .count()
)

fmm_main_lmp_interval_count = (
    main_lmp_df
    .filter(
        F.col("market_type") == "FMM"
    )
    .select("interval_start_utc")
    .distinct()
    .count()
)


test_results = {
    "dam_archive_count":
        dam_archive_count,
    "fmm_archive_count":
        fmm_archive_count,
    "dam_interval_count":
        dam_interval_count,
    "fmm_interval_count":
        fmm_interval_count,
    "invalid_timestamp_count":
        invalid_timestamp_count,
    "invalid_price_count":
        invalid_price_count,
    "duplicate_hash_count":
        duplicate_hash_count,
    "duplicate_business_key_count":
        duplicate_business_key_count,
    "invalid_duration_count":
        invalid_duration_count,
    "dam_main_lmp_interval_count":
        dam_main_lmp_interval_count,
    "fmm_main_lmp_interval_count":
        fmm_main_lmp_interval_count,
}


for test_name, test_value in (
    test_results.items()
):
    print(
        f"{test_name}: {test_value}"
    )


test_errors = []


if (
    dam_archive_count
    != EXPECTED_DAM_ARCHIVES
):
    test_errors.append(
        "DAM powinno mieć "
        f"{EXPECTED_DAM_ARCHIVES} archiwów."
    )

if (
    fmm_archive_count
    != EXPECTED_FMM_ARCHIVES
):
    test_errors.append(
        "FMM powinno mieć "
        f"{EXPECTED_FMM_ARCHIVES} archiwów."
    )

if (
    dam_interval_count
    != EXPECTED_DAM_INTERVALS
):
    test_errors.append(
        "DAM powinno mieć "
        f"{EXPECTED_DAM_INTERVALS} "
        "interwałów godzinowych."
    )

if (
    fmm_interval_count
    != EXPECTED_FMM_INTERVALS
):
    test_errors.append(
        "FMM powinno mieć "
        f"{EXPECTED_FMM_INTERVALS} "
        "interwałów 15-minutowych."
    )

if invalid_timestamp_count != 0:
    test_errors.append(
        "Występują niepoprawne timestampy."
    )

if invalid_price_count != 0:
    test_errors.append(
        "Występują niepoprawne ceny."
    )

if duplicate_hash_count != 0:
    test_errors.append(
        "Występują duplikaty hashy."
    )

if duplicate_business_key_count != 0:
    test_errors.append(
        "Występują duplikaty klucza "
        "biznesowego."
    )

if invalid_duration_count != 0:
    test_errors.append(
        "Występują niepoprawne "
        "długości interwałów."
    )

if (
    dam_main_lmp_interval_count
        != EXPECTED_DAM_INTERVALS
):
    test_errors.append(
        "Nie każdy interwał DAM "
        "ma LMP_PRC."
    )

if (
    fmm_main_lmp_interval_count
        != EXPECTED_FMM_INTERVALS
):
    test_errors.append(
        "Nie każdy interwał FMM "
        "ma LMP_PRC."
    )


if test_errors:
    print()
    print("TESTY NIEZALICZONE:")

    for error in test_errors:
        print(f"- {error}")

    raise RuntimeError(
        "Kontrola Bronze CAISO "
        "nie została zaliczona."
    )


print()
print(
    "Wszystkie testy Bronze CAISO "
    "zostały zaliczone."
)

###9) Podgląd głównej ceny LMP

In [0]:
main_lmp_preview_df = (
    caiso_table_df
    .filter(
        F.upper(
            F.col("xml_data_item")
        ) == "LMP_PRC"
    )
    .select(
        "market_type",
        "market_run_id",
        "interval_start_utc",
        "interval_end_utc",
        "node_id",
        "price_value",
        "_source_file_name"
    )
    .orderBy(
        "interval_start_utc",
        "market_type"
    )
)


display(
    main_lmp_preview_df.limit(40)
)